In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import f_regression, mutual_info_regression, RFE
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. โหลดข้อมูลและตัด Benchmark / ID / Deterministic Columns ออก
data_raw = pd.read_csv("BankChurners.csv") # หรือ path ที่ดาวน์โหลด
cols_to_drop = [
    'CLIENTNUM',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
    'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2',
    # **สำคัญมาก**: ตัด Avg_Open_To_Buy ออกทันที เพราะ Avg_Open_To_Buy = Credit_Limit - Total_Revolving_Bal
    'Avg_Open_To_Buy'
]
data = data_raw.drop(columns=cols_to_drop)

# แปลง Binary Categorical
data['Attrition_Flag'] = data['Attrition_Flag'].map({'Existing Customer': 0, 'Attrited Customer': 1})
data['Gender'] = data['Gender'].map({'M': 0, 'F': 1})

# จัดการ Ordinal Categorical
income_order = ['Less than $40K', '$40K - $60K', '$60K - $80K', '$80K - $120K', '$120K +']
data['Income_Category'] = pd.Categorical(data['Income_Category'], categories=income_order, ordered=True).codes
data['Income_Category'] = data['Income_Category'].replace(-1, np.nan)

education_order = ['Uneducated', 'High School', 'College', 'Graduate', 'Post-Graduate', 'Doctorate']
data['Education_Level'] = pd.Categorical(data['Education_Level'], categories=education_order, ordered=True).codes
data['Education_Level'] = data['Education_Level'].replace(-1, np.nan)

data['Marital_Status'] = data['Marital_Status'].replace('Unknown', np.nan)

# 2. แยก Target (Continuous) และ Features
target_col = 'Credit_Limit'
y = data[target_col]
X = data.drop(columns=[target_col])

# 3. แบ่ง Train/Test ก่อนทำ Imputation & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. จัดการ Missing Values โดยคำนวณค่าจาก X_train เท่านั้น
med_edu = X_train['Education_Level'].median()
med_inc = X_train['Income_Category'].median()
mode_mar = X_train['Marital_Status'].mode()[0]

for df in [X_train, X_test]:
    df['Education_Level'] = df['Education_Level'].fillna(med_edu)
    df['Income_Category'] = df['Income_Category'].fillna(med_inc)
    df['Marital_Status'] = df['Marital_Status'].fillna(mode_mar)

# 5. One-Hot Encoding
ohe_cols = ['Card_Category', 'Marital_Status']
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
train_ohe = ohe.fit_transform(X_train[ohe_cols])
test_ohe = ohe.transform(X_test[ohe_cols])

X_train = pd.concat([X_train.drop(columns=ohe_cols), pd.DataFrame(train_ohe, columns=ohe.get_feature_names_out(ohe_cols), index=X_train.index)], axis=1)
X_test = pd.concat([X_test.drop(columns=ohe_cols), pd.DataFrame(test_ohe, columns=ohe.get_feature_names_out(ohe_cols), index=X_test.index)], axis=1)

# 6. Scaling เฉพาะฝั่ง Train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 7. Feature Selection สำหรับ Regression
f_scores, f_pvalues = f_regression(X_train_scaled, y_train)
mi_scores = mutual_info_regression(X_train_scaled, y_train, random_state=42)

# RFE ด้วย Ridge Regression
rfe = RFE(estimator=Ridge(), n_features_to_select=10)
rfe.fit(X_train_scaled, y_train)

feature_summary = pd.DataFrame({
    'feature': X_train.columns,
    'f_score': f_scores,
    'f_pvalue': f_pvalues,
    'mutual_info': mi_scores,
    'rfe_rank': rfe.ranking_
}).sort_values('mutual_info', ascending=False)

print("Top Features สำหรับทำนาย Credit_Limit:")
print(feature_summary.head(10))

Top Features สำหรับทำนาย Credit_Limit:
                  feature      f_score       f_pvalue  mutual_info  rfe_rank
15  Avg_Utilization_Ratio  2432.601629   0.000000e+00     0.762319         1
5         Income_Category  3818.741260   0.000000e+00     0.213272         1
2                  Gender  1793.463987   0.000000e+00     0.129553         1
10    Total_Revolving_Bal    22.299996   2.371283e-06     0.126860         1
16     Card_Category_Blue  2860.832605   0.000000e+00     0.116680         1
12        Total_Trans_Amt   242.378319   7.156694e-54     0.107190         1
19   Card_Category_Silver  1863.773769   0.000000e+00     0.084496         1
13         Total_Trans_Ct    46.951240   7.810232e-12     0.039095         5
17     Card_Category_Gold   519.128770  1.972602e-111     0.035925         1
3         Dependent_count    40.271042   2.329421e-10     0.015016         2
